This week we will be looking at fundamentals of working with AI agents. First, we will look at the LLM client (here we use OpenAI). 

## Setting up the OpenAI client

The OpenAI client requires an API key and the base url of the model provider:

In [1]:
from openai import OpenAI
from atlas.vault import get_secret

MODEL_NAME = "deepseek-flash"
BASE_URL = "https://api.deepseek.com"
API_KEY = get_secret("DEEPSEEK_API_KEY")
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

### Our first LLM call

Recall that behind the API is a language model that performs text completion. Here we choose a specific model under `model` then the **message chain** is passed to that model as prompt. The first message is expected to be the **system prompt** with role `system`. Then, we can append our messages which has role `user`. 

Next, we have further configuration like whether we stream content or not, and whether the model will use **reasoning** to respond along with a *guidance* on how much **reasoning effort** the model should use in its response (generally, the higher the reasoning, the better).

In [8]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Tell me a fun fact."},
    ],
    stream=False,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}},
)

print(response.choices[0].message.content)


A fun fact: **Sharks are older than trees.** Sharks have been around for roughly 450 million years, while the first trees appeared about 385 million years ago—so sharks were swimming around long before anything tree-like existed on land. 🦈🌳


Notice that the output is markdown, so let's render it:

In [9]:
from IPython.display import display, Markdown
display(Markdown("> " + response.choices[0].message.content))

> A fun fact: **Sharks are older than trees.** Sharks have been around for roughly 450 million years, while the first trees appeared about 385 million years ago—so sharks were swimming around long before anything tree-like existed on land. 🦈🌳

For testing we can set `n > 0` in `client.chat.completions.create` to get a sample of `n` responses for the same prompt. But by default `n=1`. Hence, `response.choices` is a singleton and `[0]` just gets the sole response. Let's extract the reasoning trace:

In [10]:
from rich import print_json

print_json(response.choices[0].model_dump_json())
display(Markdown("> " + response.choices[0].model_dump()["message"]["reasoning_content"]))

{
  "finish_reason": "stop",
  "index": 0,
  "logprobs": null,
  "message": {
    "content": "A fun fact: **Sharks are older than trees.** Sharks have been around for roughly 450 million years, while the first trees appeared about 385 million years ago—so sharks were swimming around long before anything tree-like existed on land. 🦈🌳",
    "refusal": null,
    "role": "assistant",
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": null,
    "reasoning_content": "We need answer user asks fun fact. Need provide fun fact. Simple. Could choose interesting. Need no tools. Ensure accurate. Fun fact: Honey never spoils? Archaeologists found edible honey in ancient Egyptian tombs; due low moisture, acidity, hydrogen peroxide. This is common but maybe oversimplified. Or octopuses have three hearts and blue blood. Fun fact: Wombat cube poop. Need concise. \"A group of flamingos is called a flamboyance.\" Fun. Provide one fun fact. Maybe surprising: \"Sharks existed before trees.\" That's true-ish: sharks ~450 million years, trees ~385 million. Fun. Or \"Bananas are berries, strawberries aren't.\" Good. Need answer. Final."
  }
}

> We need answer user asks fun fact. Need provide fun fact. Simple. Could choose interesting. Need no tools. Ensure accurate. Fun fact: Honey never spoils? Archaeologists found edible honey in ancient Egyptian tombs; due low moisture, acidity, hydrogen peroxide. This is common but maybe oversimplified. Or octopuses have three hearts and blue blood. Fun fact: Wombat cube poop. Need concise. "A group of flamingos is called a flamboyance." Fun. Provide one fun fact. Maybe surprising: "Sharks existed before trees." That's true-ish: sharks ~450 million years, trees ~385 million. Fun. Or "Bananas are berries, strawberries aren't." Good. Need answer. Final.

Here we also see the role of the model is `assistant`.

The following examples repeat the same request and display steps, so let's put them in a function. `chat` accepts the message history and returns the full response for chaining. Its keyword arguments expose the model, reasoning effort and thinking mode; `single_paragraph` only changes how the answer is displayed.


In [18]:
def chat(
    messages,
    *,
    model=MODEL_NAME,
    reasoning_effort="high",
    thinking_type="enabled"
):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False,
        reasoning_effort=reasoning_effort,
        extra_body={"thinking": {"type": thinking_type}},
    )

    # expose reasoning trace and response content as markdown
    message = response.choices[0].message
    reasoning = message.model_dump().get("reasoning_content").replace("\n", " ")
    if reasoning:
        print("Reasoning:")
        display(Markdown("> " + reasoning))

    content = message.content or ""
    content = content.replace("\n", " ")
    print("Response:")
    display(Markdown("> " + content))
    return response

### Chaining messages

Chaining messages allows us to basically converse with the LLM. Let's make the same message as above:

In [19]:
SYSTEM_PROMPT = "You are a helpful assistant"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Tell me a fun fact."},
]

# First call
response = chat(messages)

Reasoning:


> We need answer fun fact. Need concise. Pick interesting. Need no tools. Maybe "octopuses have three hearts and blue blood" but common. Fun fact about wombat cube poop. Or "bananas are berries, strawberries aren't" common. Need fun. Could say: "A group of flamingos is called a flamboyance." Or "There are more possible chess games than atoms in observable universe" maybe. Let's craft one fun fact with brief explanation. Maybe "Wombats produce cube-shaped poop—their intestines have varying elasticity that molds feces into cubes, helping it stay put as territory markers." That's fun. Answer.

Response:


> Fun fact: Wombats poop cubes. Their intestines have uneven elasticity that molds the poop into cube shapes—handy for marking territory because the cubes don’t roll away.

Let's create a user response and chain it to the message history:

In [20]:
follow_up = "That's cool!"

msg = response.choices[0].message
messages.append({"role": msg.role, "content": msg.content})
messages.append({"role": "user", "content": follow_up})

# Second call
response = chat(messages)


Reasoning:


> We need to respond to user's enthusiasm. They said "That's cool!" after wombat fact. Need likely continue friendly, maybe add another fun fact or acknowledge. Keep concise. Could say glad you liked it, and offer another related fun fact? Desired oververbosity 5, but simple. Maybe respond with another fun fact about wombats or animals. Should not overdo. We can say "Right? Nature is weirdly efficient. If you want another: wombat pouches face backward..." That's a good follow-up. Keep brief.

Response:


> Right? Nature is weirdly efficient. Bonus wombat fact: their pouches face backward, so when they dig, dirt doesn’t get shoveled into the pouch with the baby.

The response of the model is nice and conversational due to maintaining the message history.

### Templating

Templating lets us reuse the same instructions with different inputs. A Python **f-string** can insert a topic and audience into a prompt. Let's send that prompt using the client we set up earlier:


In [26]:
def explain(topic: str, audience: str, max_words: int = 100):
    SYSTEM_PROMPT = "You are a helpful assistant"
    prompt = (
        f"Explain {topic} to {audience}. "
        f"Use one concrete example and keep the answer under {max_words} words."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    print("Prompt:")
    display(Markdown("> " + prompt))
    return chat(messages)


explain(topic="AI agents", audience="a Python beginner");

Prompt:


> Explain AI agents to a Python beginner. Use one concrete example and keep the answer under 100 words.

Reasoning:


> We need answer under 100 words, explain AI agents to Python beginner with one concrete example. Need concise. Need count words. We can say: AI agent = program that uses LLM to decide steps and call tools until goal. Example: Python script with a "weather" tool; user asks "Should I pack an umbrella in Paris?" Agent decides to call weather API, reads result, then answers. You write tool function and loop; LLM chooses actions. Keep under 100. Count words. Need maybe 70 words. Let's craft exactly. "An AI agent is a program that uses a language model to choose actions—like calling functions or APIs—until it completes a goal. Example: a Python script gives the agent a `get_weather(city)` function. User asks, 'Should I pack an umbrella in Paris?' The agent decides to call `get_weather('Paris')`, reads the result, and answers. You provide tools; the model decides when to use them." Count: An1 AI2 agent3 is4 a5 program6 that7 uses8 a9 language10 model11 to12 choose13 actions—like14 calling15 functions16 or17 APIs—until18 it19 completes20 a21 goal22. Example23: a24 Python25 script26 gives27 the28 agent29 a30 get_weather(city)31 function32. User33 asks34, Should35 I36 pack37 an38 umbrella39 in40 Paris41? The42 agent43 decides44 to45 call46 get_weather('Paris')47, reads48 the49 result50, and51 answers52. You53 provide54 tools55; the56 model57 decides58 when59 to60 use61 them62. Under 100. Good.

Response:


> An AI agent is a program that uses a language model to choose actions—like calling functions or APIs—until it completes a goal. Example: a Python script gives the agent a `get_weather(city)` function. User asks, “Should I pack an umbrella in Paris?” The agent decides to call `get_weather('Paris')`, reads the result, and answers. You provide tools; the model decides when to use them.

That seems like a pretty good explanation of AI agents for beginners. Changing `topic` or `audience` and rerunning the cell gives us a new request with the same instructions. Python fills in `{topic}` and `{audience}` before the API call; the model receives the completed text.


In [27]:
explain(topic="AI agents", audience="a mathematician");

Prompt:


> Explain AI agents to a mathematician. Use one concrete example and keep the answer under 100 words.

Reasoning:


> We need answer to user: explain AI agents to mathematician, one concrete example, under 100 words. Need concise. Mathematician framing. Need define agent as system using LLM? AI agents: perceive state, choose actions via policy, pursue objective, possibly tools/memory. Mathematician: an agent is a policy π mapping observations to actions, coupled to environment, optimizing expected return; LLM provides flexible policy. Concrete example: travel booking agent? Maybe math problem solver? One concrete example under 100 words. Need ensure under 100 words. Count. Draft:  "An AI agent is a policy π: observations → actions, interacting with an environment to maximize expected return. Unlike a single model call, it loops: observe, decide, act, update. Example: a theorem-proving agent receives a goal, proposes a proof step, calls a proof checker, reads errors, and iterates until verified or budget exhausted. Formally, it’s a controlled Markov process; the LLM supplies a flexible, learned policy and tools extend its action space."  Count words: An1 AI2 agent3 is4 a5 policy6 π:7 observations8 →9 actions,10 interacting11 with12 an13 environment14 to15 maximize16 expected17 return.18 Unlike19 a20 single21 model22 call,23 it24 loops:25 observe,26 decide,27 act,28 update.29 Example:30 a31 theorem-proving32 agent33 receives34 a35 goal,36 proposes37 a38 proof39 step,40 calls41 a42 proof43 checker,44 reads45 errors,46 and47 iterates48 until49 verified50 or51 budget52 exhausted.53 Formally,54 it’s55 a56 controlled57 Markov58 process;59 the60 LLM61 supplies62 a63 flexible,64 learned65 policy66 and67 tools68 extend69 its70 action71 space.72 Under 100. Good. Need final only.

Response:


> An AI agent is a policy π: observations → actions, interacting with an environment to maximize expected return. Unlike a single model call, it loops: observe, decide, act, update. Example: a theorem-proving agent receives a goal, proposes a proof step, calls a proof checker, reads errors, and iterates until verified or budget exhausted. Formally, it’s a controlled Markov process; the LLM supplies a flexible learned policy, and tools extend its action space.

Interestingly, the reasoning model is indeed counting. :) 

## What is an AI agent?

AI agents = AI systems that can do work autonomously
= llm controls workflow

agent = LLM with tools in a loop to achieve a goal
